# Exploratory & Market Analysis
### Project Objectives & Analytical Questions
1. **Portfolio Concentration:** How concentrated is the pharmaceutical product catalog among top-tier manufacturers versus smaller producers? Do top manufacturers dominate complex dosage forms?
2. **Price Dispersion:** Controlling for identical generic molecule, dosage form and strength, how much price dispersion exists across competing brands? Do top-tier manufacturers consistently charge a brand premium?
3. **Market Gaps:** Which clinical indications and drug classes are hyper-saturated (high brand-to-generic saturation ratio) versus underserved (low brand competition / unmanufactured generic gaps)?

In [1]:
import pandas as pd
import numpy as np

data_dir = '../data/processed'
df = pd.read_csv(f'{data_dir}/master_medicines.csv')
unmanu_generics = pd.read_csv(f'{data_dir}/unmanufactured_generics.csv')

# print(df['type'].unique())

kpis = {
    'Total Medicine Brands': len(df),
    'Allopathic Brands': (df['type'] == 'allopathic').sum(),
    'Herbal Brands': (df['type'] == 'herbal').sum(),
    'Active Manufacturers': df['manufacturer'].nunique(),
    'Active Generic Molecules': df['generic'].nunique(),
    'Unmanufactured Generics (Market Gaps)': len(unmanu_generics),
    'Unique Dosage Forms': df['dosage_form'].nunique(),
    'Unique Clinical Indications': df['indication'].nunique(),
    'Unique Drug Classes': df['drug_class'].nunique()
}

kpi_summary = pd.Series(kpis, name='Count').to_frame()
print("BANGLADESH PHARMA MARKET BASELINE")
display(kpi_summary)

display(df.head())

BANGLADESH PHARMA MARKET BASELINE


,Count
Total Medicine Brands,21712
Allopathic Brands,21361
Herbal Brands,351
Active Manufacturers,232
Active Generic Molecules,1661
Unmanufactured Generics (Market Gaps),50
Unique Dosage Forms,112
Unique Clinical Indications,660
Unique Drug Classes,358


,brand_id,brand_name,type,slug,dosage_form,generic,strength,manufacturer,package_container,price_bdt,pricing_basis,generic_id,drug_class,indication
0,4077,A-Cold,allopathic,a-coldsyrup4-mg5-ml,Syrup,Bromhexine Hydrochloride,4 mg/5 ml,ACME Laboratories Ltd.,100 ml bottle: ৳ 40.12,40.12,per_container,145,Cough expectorants & mucolytics,Bronchitis
1,4006,A-Cof,allopathic,a-cofsyrup10-mg30-mg125-mg5-ml,Syrup,Dextromethorphan + Pseudoephedrine + Triprolidine,(10 mg+30 mg+1.25 mg)/5 ml,ACME Laboratories Ltd.,100 ml bottle: ৳ 100.00,100.00,per_container,346,Combined cough suppressants,Non-productive cough
2,6174,A-Clox,allopathic,a-cloxinjection500-mgvial,Injection,Cloxacillin Sodium,500 mg/vial,ACME Laboratories Ltd.,"500 mg vial: ৳ 28.43,(5's pack: ৳ 142.15),",28.43,per_container,292,Penicillinase-resistant penicillins,UTI caused by Staphylococcus Aureus
3,6173,A-Clox,allopathic,a-cloxinjection250-mgvial,Injection,Cloxacillin Sodium,250 mg/vial,ACME Laboratories Ltd.,"250 mg vial: ৳ 20.00,(5's pack: ৳ 100.00),",20.00,per_container,292,Penicillinase-resistant penicillins,UTI caused by Staphylococcus Aureus
4,6172,A-Clox,allopathic,a-cloxpowder-for-suspension125-mg5-ml,Powder for Suspension,Cloxacillin Sodium,125 mg/5 ml,ACME Laboratories Ltd.,100 ml bottle: ৳ 45.00,45.00,per_container,292,Penicillinase-resistant penicillins,UTI caused by Staphylococcus Aureus


---
## Question 1: Portfolio Concentration
Evaluate how much of the national drug catalog is controlled by the Top 10 industry giants versus smaller manufacturers and test whether dosage form complexity acts as a barrier to entry.

### Manufacturer Portfolio Concentration (Top 10 vs. Market)

In [2]:
total_brands = df['brand_id'].nunique()

# display(df[df['brand_name']=='Napa'][['brand_id', 'brand_name', 'dosage_form', 'strength']])

mfg_summary = (
    df.groupby('manufacturer')
    .agg(
        brand_count=('brand_id', 'count'),
        unique_brand_names=('brand_name', 'nunique'),
        unique_generics=('generic', 'nunique'),
        unique_dosage_forms=('dosage_form', 'nunique')
    )
    .sort_values(by='brand_count', ascending=False)
    .reset_index()
)

top_10 = mfg_summary.head(10)
top_10_share = top_10['brand_count'].sum() / total_brands * 100

print(f"Top 10 manufacturers account for {top_10_share:.2f}% of all brand records.\n")
display(top_10)

Top 10 manufacturers account for 34.09% of all brand records.



,manufacturer,brand_count,unique_brand_names,unique_generics,unique_dosage_forms
0,Incepta Pharmaceuticals Ltd.,1182,704,689,76
1,Square Pharmaceuticals Ltd.,902,582,546,67
2,Opsonin Pharma Ltd.,900,489,462,60
3,Beximco Pharmaceuticals Ltd.,699,417,385,53
4,ACME Laboratories Ltd.,692,431,410,52
5,ACI Limited,681,371,350,55
6,Drug International Ltd.,677,425,418,46
7,Eskayef Pharmaceuticals Ltd.,657,391,360,49
8,Aristopharma Ltd.,512,315,311,56
9,Renata Limited,499,295,273,40


### Market Structure by Manufacturer Tier
Group all the manufacturers into 3 tiers based on portfolio rank: 

Top 10 Giants, Mid-Tier Challengers, Small / Niche Producers

In [3]:
print(f"Total Manufactureres: {df['manufacturer'].nunique()}")

conditions = [
    mfg_summary.index < 10,
    mfg_summary.index < 50
]
choices = ['1. Top 10 Giants', '2. Mid-Tier (Rank 11-50)']

mfg_summary['mfg_tier'] = np.select(conditions, choices, default='3. Small/Niche (Rank 51-232)')

tier_overview = (
    mfg_summary.groupby('mfg_tier')
    .agg(
        manufacturer_count=('manufacturer', 'count'),
        total_brands=('brand_count', 'sum'),
        median_brands_per_mfg=('brand_count', 'median'),
        median_generics_per_mfg=('unique_generics', 'median')
    )
    .reset_index()
)

# display(tier_overview)

tier_overview['total_brand_share_pct'] = (tier_overview['total_brands'] / total_brands * 100).round(2)
display(tier_overview)

Total Manufactureres: 232


,mfg_tier,manufacturer_count,total_brands,median_brands_per_mfg,median_generics_per_mfg,total_brand_share_pct
0,1. Top 10 Giants,10,7401,686.5,397.5,34.09
1,2. Mid-Tier (Rank 11-50),40,8707,161.5,103.0,40.10
2,3. Small/Niche (Rank 51-232),182,5604,14.5,10.0,25.81


### Dosage Form Complexity & Manufacturing Barriers by Tier

In [4]:
# Cell 4: Map mfg_tier to df and check dosage form share across tiers
# df = df.merge(
#     mfg_summary[['manufacturer', 'mfg_tier']], 
#     on='manufacturer', 
#     how='left'
# )
df['mfg_tier'] = df['manufacturer'].map(
    mfg_summary.set_index('manufacturer')['mfg_tier']
)

# display(df.head())

# # Calculate percentage split across tiers for each dosage form
dosage_tier_pct = pd.crosstab(
    df['dosage_form'],
    df['mfg_tier'],
    normalize='index'
) * 100

dosage_tier_pct['total_brands'] = df['dosage_form'].value_counts()

display(dosage_tier_pct.head())

# # # Filter to major dosage forms (>= 100 brands) and sort by Top 10 Giants' share
major_dosage_forms = (
    dosage_tier_pct[dosage_tier_pct['total_brands'] >= 100]
    .sort_values(by='1. Top 10 Giants', ascending=False)
    .round(1)
)

display(major_dosage_forms.head(10))
print("\n ----- Lower barrier to entry -----")
display(major_dosage_forms.tail(10))

mfg_tier,1. Top 10 Giants,2. Mid-Tier (Rank 11-50),3. Small/Niche (Rank 51-232),total_brands
dosage_form,,,,
Capsule,31.046570,38.858287,30.095143,1997
Capsule (Controlled Release),25.000000,50.000000,25.000000,4
Capsule (Delayed Release),18.764302,40.732265,40.503432,437
Capsule (Extended Release),33.333333,66.666667,0.000000,9
Capsule (Modified Release),27.777778,50.000000,22.222222,36


mfg_tier,1. Top 10 Giants,2. Mid-Tier (Rank 11-50),3. Small/Niche (Rank 51-232),total_brands
dosage_form,,,,
Inhalation Capsule,71.6,22.0,6.4,109
SC Injection,54.0,24.4,21.6,213
Tablet (Extended Release),53.4,39.7,6.9,247
IM/IV Injection,45.9,43.3,10.8,610
Tablet (Sustained Release),44.9,37.8,17.3,127
Ointment,44.2,38.0,17.8,242
Ophthalmic Suspension,42.2,50.0,7.8,116
Pediatric Drops,42.2,37.3,20.5,244
Cream,41.2,37.6,21.2,495



 ----- Lower barrier to entry -----


mfg_tier,1. Top 10 Giants,2. Mid-Tier (Rank 11-50),3. Small/Niche (Rank 51-232),total_brands
dosage_form,,,,
Tablet (Delayed Release),30.3,44.3,25.4,122
IM Injection,28.7,50.5,20.7,275
Syrup,27.4,37.8,34.7,1002
Chewable Tablet,25.8,41.7,32.5,283
Injection,24.3,48.1,27.5,189
Oral Suspension,24.3,37.1,38.6,568
Powder for Suspension,23.2,41.7,35.1,1004
Capsule (Delayed Release),18.8,40.7,40.5,437
Tablet (Enteric Coated),18.2,46.6,35.2,307


#### Key Findings: Market Concentration & Manufacturing Barriers (Question 1)
* **Observation:** The top 10 manufacturers account for 34.1% of all registered brands and the top 50 companies control 74.2% of the catalog. Meanwhile, 182 small/niche companies share the remaining 25.8%.
* **Insight:** Dosage form, Top 10 giants dominate at Inhalation Capsules (71.6% share), Extended-Release Tablets (53.4%), and IM/IV Injections (45.9%), where small companies hold less than 11% share. Conversely, small companies crowd into lower-barrier liquids (Syrups 34.7%, Oral Suspensions 38.6%) and Delayed/Timed-Release Capsules (40.5%–55.6%).
* **Business Recommendation:** A mid-tier manufacturer looking to escape fierce price wars with 180+ small competitors should avoid launching more standard syrups or basic gastric capsules. Instead invest in Ophthalmic, Injectable or Inhalation lines where competition is restricted to the top 10–50 players.

---
## Question 2: Price Dispersion
Measure price spread across identical medicines (same generic, dosage form, strength and pricing basis) and test whether Top 10 manufacturers charge a premium over smaller competitors.

### Price dispersion across identical medicine cohorts

In [11]:
# print(df.isnull().sum())
# filtered_allopathic = df[(df['type'] == 'allopathic') & (df['price_bdt'].notna())].copy()
# Container size can differ. So only considering per_unit pricing.
filtered_allopathic = df[
    (df['type'] == 'allopathic') & 
    (df['pricing_basis'] == 'per_unit') & 
    (df['price_bdt'].notna())
].copy()

cohort_pricing = (
    filtered_allopathic.groupby(['generic', 'dosage_form', 'strength', 'pricing_basis'])
    .agg(
        competing_brands=('brand_id', 'count'),
        min_price=('price_bdt', 'min'),
        q25_price=('price_bdt', lambda x: x.quantile(0.25)),
        median_price=('price_bdt', 'median'),
        q75_price=('price_bdt', lambda x: x.quantile(0.75)),
        max_price=('price_bdt', 'max')
    )
    .reset_index()
)

# Filter to cohorts with at least 10 competing brands
cohort_pricing = cohort_pricing[cohort_pricing['competing_brands'] >= 10].copy()

# Calculate max-to-min ratio and interquartile ratio (Q75 / Q25)
cohort_pricing['max_to_min_ratio'] = (cohort_pricing['max_price'] / cohort_pricing['min_price']).round(2)
cohort_pricing['iqr_ratio'] = (cohort_pricing['q75_price'] / cohort_pricing['q25_price']).round(2)

print(f"Total comparable unit-dose cohorts (>= 10 brands): {len(cohort_pricing)}")
print(f"Median max-to-min price spread: {cohort_pricing['max_to_min_ratio'].median():.2f}x")
print(f"Median middle-50% price spread (Q75/Q25): {cohort_pricing['iqr_ratio'].median():.2f}x\n")

print("Top 10 Unit-Dose Cohorts with the Highest Price Disparity:")
display(cohort_pricing.sort_values(by='max_to_min_ratio', ascending=False).head(10))

Total comparable unit-dose cohorts (>= 10 brands): 347
Median max-to-min price spread: 1.76x
Median middle-50% price spread (Q75/Q25): 1.09x

Top 10 Unit-Dose Cohorts with the Highest Price Disparity:


,generic,dosage_form,strength,pricing_basis,competing_brands,min_price,q25_price,median_price,q75_price,max_price,max_to_min_ratio,iqr_ratio
444,Diclofenac Sodium,Tablet,50 mg,per_unit,38,0.40,0.7,0.805,0.8725,8.50,21.25,1.25
1292,Rivastigmine Tartrate,Capsule,1.5 mg,per_unit,10,10.00,20.0,21.500,23.0000,145.55,14.56,1.15
896,Metformin Hydrochloride,Tablet,500 mg,per_unit,70,1.10,3.0,4.000,4.0000,16.00,14.55,1.33
148,Azithromycin Dihydrate,Tablet,500 mg,per_unit,114,4.00,30.0,35.000,35.0000,55.50,13.88,1.17
748,Ketorolac Tromethamine,Tablet,10 mg,per_unit,108,4.00,10.0,10.000,10.0325,45.00,11.25,1.00
1024,Nitazoxanide,Tablet,500 mg,per_unit,35,9.00,10.0,10.030,10.0500,100.00,11.11,1.01
782,Letrozole,Tablet,2.5 mg,per_unit,15,40.00,40.0,40.000,40.0600,376.42,9.41,1.00
973,Multivitamin & Multimineral,Tablet,Not Specified,per_unit,44,1.00,1.5,1.500,1.8025,9.00,9.00,1.20
445,Diclofenac Sodium,Tablet (Sustained Release),100 mg,per_unit,18,2.52,3.0,3.005,3.0100,18.00,7.14,1.00
1305,Rosuvastatin,Tablet,20 mg,per_unit,23,8.00,30.0,30.000,30.0000,50.00,6.25,1.00


### Premium Price Analysis by Manufacturer Tier

In [16]:
cohort_lookup = cohort_pricing[['generic', 'dosage_form', 'strength', 'median_price']]

unit_brands_compared = filtered_allopathic.merge(
    cohort_lookup,
    on=['generic', 'dosage_form', 'strength'],
    how='inner'
)

# display(unit_brands_compared.head())

# Calculate how each brand prices relative to the market median for that exact pill
unit_brands_compared['price_to_median_ratio'] = unit_brands_compared['price_bdt'] / unit_brands_compared['median_price']

# Classify whether a brand is priced Above, At or Below the cohort median
# 1% tolerance
conditions = [
    unit_brands_compared['price_to_median_ratio'] > 1.01,
    unit_brands_compared['price_to_median_ratio'] < 0.99
]
choices = ['Above Median (>1.01x)', 'Below Median (<0.99x)']
unit_brands_compared['pricing_stance'] = np.select(conditions, choices, default='At Market Median (1.00x)')

# display(unit_brands_compared.head())

# Compare pricing stance across manufacturer tiers
tier_pricing_stance = pd.crosstab(
    unit_brands_compared['mfg_tier'], 
    unit_brands_compared['pricing_stance'], 
    normalize='index'
) * 100

# Also add mean and median price_to_median_ratio per tier
tier_ratio_stats = unit_brands_compared.groupby('mfg_tier')['price_to_median_ratio'].agg(
    median_ratio='median',
    mean_ratio='mean'
)

tier_pricing_summary = tier_ratio_stats.join(tier_pricing_stance).round(2)
display(tier_pricing_summary)

,median_ratio,mean_ratio,Above Median (>1.01x),At Market Median (1.00x),Below Median (<0.99x)
mfg_tier,,,,,
1. Top 10 Giants,1.0,1.03,25.58,61.36,13.06
2. Mid-Tier (Rank 11-50),1.0,1.01,22.26,53.76,23.98
3. Small/Niche (Rank 51-232),1.0,1.00,20.07,43.68,36.25


### Key Findings: Price Dispersion & Brand Premium (Question 2)
* **Observation:** Across 347 comparable unit-dose drug cohorts ($\ge 10$ brands), the extreme price spread (max_to_min_ratio) has a median of 1.76x (with isolated cohorts reaching 13x–21x due to imported multinational brands vs locals). However, the core middle-50% of the market has a median spread of just 1.09x (only a 9% price difference).
* **Insight (Tier Pricing Strategy):** Top 10 Giants act as the market's price anchor. 86.9% of their unit-dose products are priced At (61.4%) or Above (25.6%) the cohort median and only 13.1% are discounted below median. Conversely, Small/Niche manufacturers price 36.3% of their products below the market median, nearly 3x the discounting rate of Top 10 giants.
* **Business Implication:** Because DGDA Maximum Retail Price (MRP) ceilings tightly compress the middle 50% of prices (1.09x IQR), top-tier manufacturers cannot rely on massive unit-price premiums on standard generics. Instead they capture margin by pricing consistently at the regulatory ceiling while forcing smaller competitors to discount (<0.99x) to compete.

---
## Question 3: Market Gaps
Identify which drug classes are hyper-saturated (Red Oceans) versus underserved (White Space) using the Brand-to-Generic Saturation Ratio, and evaluate unmanufactured/monopoly molecules for portfolio expansion.

### Hyper-Saturated Drug Classes (Red Oceans)

In [73]:
# display(df.info())

drug_saturation = (
    df[df['drug_class'] != 'Not Specified']
    .groupby('drug_class')
    .agg(
        total_brands=('brand_id', 'count'),
        unique_generics=('generic', 'nunique'),
        competing_mfgs=('manufacturer', 'nunique')
    )
    .reset_index()
)

drug_saturation['brands_per_generic'] = (
    drug_saturation['total_brands'] / drug_saturation['unique_generics']
).round(2)

# display(drug_saturation)

filtered_drug_saturation = (
    drug_saturation[drug_saturation['unique_generics'] >= 5]
    .sort_values(by='brands_per_generic', ascending=False)
)

display(filtered_drug_saturation.head(10))

,drug_class,total_brands,unique_generics,competing_mfgs,brands_per_generic
295,Proton Pump Inhibitor,546,6,128,91.00
327,Third generation Cephalosporins,903,10,107,90.30
232,Non-sedating antihistamines,457,8,100,57.12
135,Drugs used for Rheumatoid Arthritis,215,5,111,43.00
10,Amoebicides,198,5,91,39.60
134,Drugs for subcutaneous and mycoses,268,7,93,38.29
15,Anthelmintic,183,5,86,36.60
256,Other Anti-anginal & Anti-ischaemic drugs,287,8,75,35.88
3,Adjunct anti-epileptic drugs,434,13,68,33.38
12,Angiotensin-ll receptor blocker,231,7,65,33.00


### Underserved Drug Classes (Blue Oceans)

In [74]:
underserved_drugs = (
    drug_saturation[drug_saturation['unique_generics'] >= 5]
    .sort_values(by='brands_per_generic', ascending=True)
)

display(underserved_drugs.head(10))

,drug_class,total_brands,unique_generics,competing_mfgs,brands_per_generic
318,Sunblock Preparation,8,7,5,1.14
149,Emollients & combined preparations,12,7,8,1.71
219,Miscellaneous topical agents,32,18,17,1.78
178,Herbal and Nutraceuticals,259,143,24,1.81
267,Other ophthalmic preparations,15,8,13,1.88
347,"Vaccines, Anti-sera & Immunoglobulin",70,35,18,2.00
268,Other preparations,14,6,9,2.33
246,Oral Contraceptive preparations,33,14,9,2.36
261,Other antibacterial preparation,19,8,9,2.38
229,Non depolarizing muscle relaxants,12,5,7,2.40


### Unmanufactured Generics by Drug Class (White Space)

In [75]:
display(
    unmanu_generics['drug_class']
    .value_counts()
    .head(10)
    .to_frame(name='unmanufactured_generics_count')
    .reset_index()
)

,drug_class,unmanufactured_generics_count
0,Antidote preparations,3
1,Antiparkinson drugs,2
2,Anticholinergics (antimuscarinics)/ Anti-spasm...,2
3,H2 receptor antagonist,2
4,Drugs for HIV / Anti-retroviral drugs,2
5,Anti-Tubercular Chemotherapeutics,2
6,Contrast medium for diagnostic procedures,2
7,Anti-malarial drugs,2
8,Drugs used in Vaginal and Vulval condition,2
9,Calcium Regulator,1


### Molecule-Level Competition Intensity & Pricing Power

In [76]:
generic_competition = (
    df.groupby(['generic', 'drug_class', 'indication'])
    .agg(
        total_brands=('brand_id', 'count'),
        competing_mfgs=('manufacturer', 'nunique'),
        median_price_bdt=('price_bdt', 'median')
    )
    .reset_index()
)

conditions = [
    generic_competition['competing_mfgs'] == 1,
    generic_competition['competing_mfgs'] == 2,
    generic_competition['competing_mfgs'].between(3, 10)
]
choices = [
    '1. Monopoly (1 Mfg)',
    '2. Duopoly (2 Mfgs)',
    '3. Moderate (3-10 Mfgs)'
]
generic_competition['competition_tier'] = np.select(
    conditions, 
    choices, 
    default='4. Crowded (11+ Mfgs)'
)

# display(generic_competition)

competition_summary = (
    generic_competition.groupby('competition_tier')
    .agg(
        generic_count=('generic', 'count'),
        total_brands=('total_brands', 'sum'),
        median_drug_price=('median_price_bdt', 'median')
    )
    .round(2)
)
competition_summary['share_of_generics_pct'] = (
    competition_summary['generic_count'] / len(generic_competition) * 100
).round(2)

competition_summary['share_of_brands_pct'] = (
    competition_summary['total_brands'] / competition_summary['total_brands'].sum() * 100
).round(2)

display(competition_summary)

,generic_count,total_brands,median_drug_price,share_of_generics_pct,share_of_brands_pct
competition_tier,,,,,
1. Monopoly (1 Mfg),542,655,77.76,32.63,3.02
2. Duopoly (2 Mfgs),199,498,60.04,11.98,2.29
3. Moderate (3-10 Mfgs),613,4490,45.00,36.91,20.68
4. Crowded (11+ Mfgs),307,16069,15.00,18.48,74.01


### Key Findings: Market Gaps (Question 3)
* **Observation:** Just 18.48% of generic molecules (307) account for 74.01% of all registered brands (16,069 brands), led by hyper-saturated Proton Pump Inhibitors (91.0 brands/generic) and 3rd-Gen Cephalosporins (90.3). Conversely, 44.61% of all active generics (741) are Monopolies (542) or Duopolies (199).
* **Insight:** Competition directly dictates pricing power. Median drug prices climb 5.2x from ৳15.00 in Crowded generics (11+ mfgs) up to ৳60.04 in Duopolies and ৳77.76 in Monopolies. Among the 50 unmanufactured generics (0 brands), infectious diseases (HIV, TB, Malaria) reflect public/NGO distribution. However, Antidotes (3), Antiparkinson (2) and Contrast Medium (2) represent true commercial gaps.
* **Business Recommendation:** Reduce launches of similar brands in highly competitive gastric and antibiotic markets, where median prices are around ৳15.00. Instead, explore opportunities in monopoly and duopoly molecules (median prices of ৳60–78), less competitive drug classes such as vaccines and contraceptives. Also, unmanufactured molecules such as diagnostic contrast media and antidotes. Prioritise investments based on market demand, production costs and commercial feasibility.

---
## Export Enriched Dataset for Visualization & BI

In [102]:
# display(df.info())
# display(unit_brands_compared.info())
df['competition_tier'] = df['generic'].map(
    generic_competition.set_index('generic')['competition_tier']
)

df['pricing_stance'] = df['brand_id'].map(
    unit_brands_compared.set_index('brand_id')['pricing_stance']
).fillna('Not Applicable (Non-Unit / <10 Brands)')

df['price_to_median_ratio'] = df['brand_id'].map(
    unit_brands_compared.set_index('brand_id')['price_to_median_ratio']
).round(4)

# Export
df.to_csv(
    '../data/processed/master_medicines_enriched.csv', 
    index=False
)

display(df[['brand_id', 'brand_name', 'manufacturer', 'mfg_tier', 'competition_tier', 'pricing_stance', 'price_to_median_ratio']]
        [df['pricing_basis']=='per_unit']
        .head(10)
)

,brand_id,brand_name,manufacturer,mfg_tier,competition_tier,pricing_stance,price_to_median_ratio
5,6171,A-Clox,ACME Laboratories Ltd.,1. Top 10 Giants,4. Crowded (11+ Mfgs),Above Median (>1.01x),1.0101
6,14006,A-Care,Asiatic Laboratories Ltd.,2. Mid-Tier (Rank 11-50),4. Crowded (11+ Mfgs),At Market Median (1.00x),0.9960
7,3039,A-Card,ACME Laboratories Ltd.,1. Top 10 Giants,3. Moderate (3-10 Mfgs),Not Applicable (Non-Unit / <10 Brands),NaN
8,15483,A-Calm,ACME Laboratories Ltd.,1. Top 10 Giants,4. Crowded (11+ Mfgs),Above Median (>1.01x),1.4963
9,14480,A-Cal DX,ACME Laboratories Ltd.,1. Top 10 Giants,4. Crowded (11+ Mfgs),Not Applicable (Non-Unit / <10 Brands),NaN
10,14399,A-Cal D,ACME Laboratories Ltd.,1. Top 10 Giants,4. Crowded (11+ Mfgs),Above Median (>1.01x),1.3000
11,14264,A-Cal,ACME Laboratories Ltd.,1. Top 10 Giants,4. Crowded (11+ Mfgs),Above Median (>1.01x),1.6667
12,13582,A-B1,ACME Laboratories Ltd.,1. Top 10 Giants,4. Crowded (11+ Mfgs),Above Median (>1.01x),1.1622
13,25154,5X,Renata Limited,1. Top 10 Giants,3. Moderate (3-10 Mfgs),Not Applicable (Non-Unit / <10 Brands),NaN
15,27661,Arubin,Square Pharmaceuticals Ltd.,1. Top 10 Giants,2. Duopoly (2 Mfgs),Not Applicable (Non-Unit / <10 Brands),NaN
